# BirdCLEF+ 2026

## Score: .50

In [ ]:
from dataclasses import dataclass, fields
from pathlib import Path
import re

import librosa
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
from tqdm.auto import tqdm


def resolve_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/birdclef-2026"),
        Path.cwd() / "birdclef-2026",
        Path.cwd().parent / "birdclef-2026",
        Path.cwd(),
    ]
    for cand in candidates:
        if cand.exists() and (cand / "sample_submission.csv").exists():
            return cand.resolve()
    raise FileNotFoundError("Could not locate birdclef-2026 data directory.")


@dataclass
class BirdCLEFConfig:
    sr: int = 32000
    chunk_duration: float = 5.0
    n_mels: int = 192
    n_fft: int = 2048
    hop_length: int = 320
    fmin: int = 20
    fmax: int = 16000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = "slaney"
    mel_scale: str = "htk"
    backbone: str = "tf_efficientnet_b1.ns_jft_in1k"
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.15
    drop_path_rate: float = 0.05
    gem_p_init: float = 3.0
    val_batch_size: int = 24
    eval_stride_seconds: float = 2.5

    @property
    def chunk_samples(self) -> int:
        return int(self.sr * self.chunk_duration)

    @property
    def eval_stride_samples(self) -> int:
        return max(1, int(self.sr * self.eval_stride_seconds))


class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg: BirdCLEFConfig):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr,
            n_fft=cfg.n_fft,
            hop_length=cfg.hop_length,
            n_mels=cfg.n_mels,
            f_min=cfg.fmin,
            f_max=cfg.fmax,
            power=cfg.power,
            norm=cfg.norm,
            mel_scale=cfg.mel_scale,
        )
        self.db = T.AmplitudeToDB(stype="power", top_db=cfg.top_db)

    def forward(self, waveforms: torch.Tensor) -> torch.Tensor:
        mel = self.mel(waveforms.float())
        mel = self.db(mel)
        mel = torch.nan_to_num(mel, nan=-80.0, posinf=0.0, neginf=-80.0)
        mel_flat = mel.flatten(1)
        mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mel_min) / (mel_max - mel_min + 1e-6)
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)


class GEMFreqPool(nn.Module):
    def __init__(self, p_init: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)


class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim: int, num_classes: int, dropout: float = 0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor):
        x = self.fc(x.permute(0, 2, 1)).permute(0, 2, 1)
        att = F.softmax(torch.tanh(self.att_conv(x)), dim=-1)
        cls = self.cls_conv(x)
        clipwise_logit = (att * cls).sum(dim=-1)
        return {"clipwise_logit": clipwise_logit, "clipwise_prob": torch.sigmoid(clipwise_logit)}


class BirdCLEFModel(nn.Module):
    def __init__(self, cfg: BirdCLEFConfig):
        super().__init__()
        self.frontend = MelSpectrogramTransform(cfg)
        self.backbone = timm.create_model(
            cfg.backbone,
            pretrained=False,
            in_chans=cfg.in_channels,
            features_only=False,
            global_pool="",
            num_classes=0,
            drop_path_rate=cfg.drop_path_rate,
        )
        self.pool = GEMFreqPool(cfg.gem_p_init)
        self.head = AttentionSEDHead(self.backbone.num_features, cfg.num_classes, cfg.dropout)

    def forward(self, waveforms: torch.Tensor):
        x = self.frontend(waveforms)
        x = self.backbone(x)
        x = self.pool(x)
        return self.head(x)


def load_checkpoint(checkpoint_path: str, device: torch.device, species_count: int):
    try:
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    except TypeError:
        ckpt = torch.load(checkpoint_path, map_location=device)

    state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    if isinstance(ckpt, dict) and "config" in ckpt:
        allowed = {f.name for f in fields(BirdCLEFConfig)}
        filtered = {k: v for k, v in ckpt["config"].items() if k in allowed}
        cfg = BirdCLEFConfig(**filtered)
    else:
        cfg = BirdCLEFConfig()
    species = ckpt.get("species", None) if isinstance(ckpt, dict) else None

    if any(k.startswith("backbone.features.") for k in state.keys()):
        raise ValueError(
            "Attached checkpoint is legacy EfficientNet format. Train/export the new SED-format checkpoint from model.ipynb and attach that instead."
        )

    if species is not None:
        cfg.num_classes = len(species)
    else:
        cfg.num_classes = species_count
    model = BirdCLEFModel(cfg).to(device)
    model.load_state_dict(state, strict=True)
    model.eval()
    return {"type": "sed", "model": model, "cfg": cfg, "species": species}


def parse_row_id(row_id: str):
    m = re.match(r"^(.*)_(\d+)$", str(row_id))
    return (m.group(1), int(m.group(2))) if m else (None, None)


def merge_windows_to_chunks(probs_windows, starts, chunk_samples, n_chunks):
    merged = np.zeros((n_chunks, probs_windows.shape[1]), dtype=np.float64)
    counts = np.zeros((n_chunks, 1), dtype=np.float64)
    for j, start in enumerate(starts):
        lo = max(0, start // chunk_samples)
        hi = min(n_chunks - 1, (start + chunk_samples - 1) // chunk_samples)
        for i in range(lo, hi + 1):
            merged[i] += probs_windows[j]
            counts[i] += 1
    return np.clip(merged / np.maximum(counts, 1), 0.0, 1.0).astype(np.float32)


def predict_submission(data_dir: Path, checkpoint_path: str, output_path: str = "submission.csv"):
    sample_sub = pd.read_csv(data_dir / "sample_submission.csv")
    expected_ids = sample_sub["row_id"].tolist()
    species = [c for c in sample_sub.columns if c != "row_id"]

    expected_by_stem = {}
    for row_id in expected_ids:
        stem, end_sec = parse_row_id(row_id)
        if stem is not None:
            expected_by_stem.setdefault(stem, []).append(end_sec)
    for stem in expected_by_stem:
        expected_by_stem[stem] = sorted(expected_by_stem[stem])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loaded = load_checkpoint(checkpoint_path, device, species_count=len(species))
    model = loaded["model"]
    cfg = loaded["cfg"]
    ckpt_species = loaded["species"]

    if ckpt_species is not None and list(ckpt_species) != list(species):
        raise ValueError("Checkpoint species order does not match sample_submission columns.")

    test_dir = data_dir / "test_soundscapes"
    train_sc_dir = data_dir / "train_soundscapes"
    soundscape_files = sorted(test_dir.glob("*.ogg")) if test_dir.exists() else []
    if not soundscape_files:
        for stem in sorted(expected_by_stem):
            p = train_sc_dir / f"{stem}.ogg"
            if p.exists():
                soundscape_files.append(p)
        if not soundscape_files and train_sc_dir.exists():
            soundscape_files = sorted(train_sc_dir.glob("*.ogg"))[:5]

    all_row_ids, all_preds = [], []
    for path in tqdm(soundscape_files, desc="Inference"):
        audio, _ = librosa.load(str(path), sr=cfg.sr, mono=True)
        audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        stem = path.stem
        if stem in expected_by_stem:
            n_chunks = max(expected_by_stem[stem]) // int(cfg.chunk_duration)
        else:
            n_chunks = max(1, len(audio) // cfg.chunk_samples)

        chunk = cfg.chunk_samples
        stride = cfg.eval_stride_samples
        padded = np.pad(audio, (0, max(0, n_chunks * chunk - len(audio))))[: n_chunks * chunk]
        starts = list(range(0, len(padded) - chunk + 1, stride))
        if not starts:
            starts = [0]
        windows = np.stack([padded[s:s + chunk] for s in starts]).astype(np.float32)

        probs_all = []
        bs = max(1, cfg.val_batch_size)
        with torch.no_grad():
            for i in range(0, len(windows), bs):
                batch = torch.from_numpy(windows[i:i + bs]).to(device)
                probs = model(batch)["clipwise_prob"].detach().cpu().numpy()
                probs_all.append(probs)
        probs_windows = np.concatenate(probs_all, axis=0)
        probs = merge_windows_to_chunks(probs_windows, starts, chunk, n_chunks)

        valid = (
            [(e // int(cfg.chunk_duration)) - 1 for e in expected_by_stem[stem]]
            if stem in expected_by_stem
            else list(range(n_chunks))
        )
        for i in valid:
            if 0 <= i < n_chunks:
                all_row_ids.append(f"{stem}_{(i + 1) * int(cfg.chunk_duration)}")
                all_preds.append(probs[i])

    if all_preds:
        pred_df = pd.DataFrame(np.asarray(all_preds, dtype=np.float32), columns=species, index=all_row_ids)
        pred_df = pred_df[~pred_df.index.duplicated(keep="first")]
    else:
        pred_df = pd.DataFrame(np.zeros((0, len(species)), dtype=np.float32), columns=species)
        pred_df.index = pd.Index([], name="row_id")

    missing = [rid for rid in expected_ids if rid not in pred_df.index]
    if missing:
        pred_df = pd.concat([
            pred_df,
            pd.DataFrame(
                np.zeros((len(missing), len(species)), dtype=np.float32),
                columns=species,
                index=pd.Index(missing, name="row_id"),
            ),
        ])

    submission = pred_df.loc[expected_ids].reset_index()
    if submission.columns[0] != "row_id":
        submission = submission.rename(columns={submission.columns[0]: "row_id"})
    submission = submission[["row_id"] + species]
    submission.to_csv(output_path, index=False)
    print("Loaded checkpoint type: sed")
    return submission


DATA_DIR = resolve_data_dir()
CHECKPOINT_PATHS = [
    "/kaggle/input/datasets/oliverlang/best-pt-sed-v1/best.pt",
    "/kaggle/input/datasets/oliverlang/best-pt/best.pt",
    "/kaggle/input/birdclef-primary-model/best.pt",
    "/kaggle/input/best-pt/best.pt",
    "models/best.pt",
    str(Path.cwd() / "models" / "best.pt"),
]


def discover_checkpoint_path():
    fixed = next((p for p in CHECKPOINT_PATHS if Path(p).exists()), None)
    if fixed is not None:
        return fixed

    kaggle_input = Path("/kaggle/input")
    discovered = []
    if kaggle_input.exists():
        discovered += sorted(kaggle_input.glob("**/best.pt"))
        discovered += sorted(kaggle_input.glob("**/*.pth"))
        discovered += sorted(kaggle_input.glob("**/*.pt"))
    for p in discovered:
        if p.name.lower() == "best.pt":
            return str(p)
    for p in discovered:
        if p.suffix.lower() in {".pt", ".pth"}:
            return str(p)
    return None


CHECKPOINT_PATH = discover_checkpoint_path()
if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        "Could not find checkpoint. Attach your trained model dataset (containing best.pt) to this Kaggle notebook."
    )

print(f"DATA_DIR={DATA_DIR}")
print(f"CHECKPOINT_PATH={CHECKPOINT_PATH}")
print(f"Device={'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
submission = predict_submission(
    data_dir=DATA_DIR,
    checkpoint_path=CHECKPOINT_PATH,
    output_path="submission.csv",
)

print(f"Saved submission.csv | shape={submission.shape}")
submission.head()

In [ ]:
# The shared helper handles Kaggle hidden test files and local validation fallback.
Path(CHECKPOINT_PATH)

In [ ]:
submission.describe().loc[["mean", "max"]]

In [ ]:
# `submission.csv` is the final Kaggle artifact.
submission.head()